In [1]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
df = tfds.load('imdb_reviews', as_supervised=True)
train, test = df['train'], df['test']

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.T4DWYQ_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.T4DWYQ_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.T4DWYQ_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


In [2]:
batch_size=32
train = train.shuffle(10000).batch(batch_size)
test = test.batch(batch_size)

In [3]:
train

<_BatchDataset element_spec=(TensorSpec(shape=(None,), dtype=tf.string, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

In [4]:
example, label = next(iter(train)) # iter() -> makes the dataset iterable,  next() -> fetches the next item from the iterator
example.numpy()[0], label.numpy()[0]

(b'Green Eyes is a great movie. In todays context of supporting our troops, it is interesting this movie showed the lack of respect soldiers received from doing their duty, during this period. From a historical view, the end of the Vietnam war left all of us with something to remember and learn from. Gene was very proud of this movie, and he deserved the credits he received from writing "Green Eyes". I agree, I do not understand why this movie is not shown more often, or at all. This movie is the kind of movie that should be shown on TV every year, much like the Wizard of Oz. The dedication of one man towards his lost son is entirely moving. I was a friend of Gene Logans and I was proud to know him. Rocky',
 np.int64(1))

In [5]:
# Text vectorization
# output_mode='int' -> each word becomes a vocabulary index,  output_sequence_length=100 -> padded to 100 tokens
vectorize_layer = tf.keras.layers.TextVectorization(output_mode='int', output_sequence_length=100)
vectorize_layer.adapt(train.map(lambda text, label: text))

In [6]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, Dense, Bidirectional, LSTM

model = Sequential([
    vectorize_layer,
    Embedding(len(vectorize_layer.get_vocabulary()), 64, mask_zero=True),
    Bidirectional(LSTM(64,  return_sequences=True)),
    Bidirectional(LSTM(32)),
    Dense(64, activation='relu'),
    Dense(1)
])

# model.build() creates and initializes weights by fixing the input shape, None allows any batch size
model.build(input_shape=(None,))

model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(),
              metrics=['accuracy'])

In [7]:
with tf.device('/GPU:0'):
  model.fit(train, epochs=3, validation_data=test)

Epoch 1/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 36s 39ms/step - accuracy: 0.7031 - loss: 0.5192 - val_accuracy: 0.7750 - val_loss: 0.4320
Epoch 2/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 36ms/step - accuracy: 0.9192 - loss: 0.2065 - val_accuracy: 0.8017 - val_loss: 0.4894
Epoch 3/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 36ms/step - accuracy: 0.9745 - loss: 0.0726 - val_accuracy: 0.7610 - val_loss: 0.8432
